<a href="https://colab.research.google.com/github/rofiqf/ReguLaw-RAG/blob/main/ReguLaw_RAG_Colab_Revisi_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline RAG — Ekstraksi Literature GAP Matrix
**Mata Kuliah:** Introduction to Generative AI (RTM 2) — Magister Artificial Intelligence, Universitas Dian Nuswantoro

**Kelompok:** Rofiq Fauzi (P36.2026.00047) & Hasan Shobri (P36.2026.00004)

Notebook ini membangun pipeline **Retrieval-Augmented Generation (RAG)** untuk mengekstrak *Literature GAP Matrix* secara otomatis dari 11 artikel ilmiah PDF (domain hukum, keuangan, kesehatan, dan industri).

**Arsitektur (6 tahap):**
1. Document Processing — `PyPDFLoader`
2. Chunking — `RecursiveCharacterTextSplitter` (chunk 1000, overlap 150)
3. Embedding lokal — `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`
4. Vector store — FAISS (in-memory)
5. Retrieval — top-k = 5 chunk paling relevan
6. Augmented Generation — `llama3-8b-8192` via **Groq API** (temperature 0.1, strict factual grounding)

**Cara pakai di Google Colab:**
1. Buka notebook ini di Colab, jalankan Sel 1 (instalasi).
2. Masukkan **Groq API key** (gratis di https://console.groq.com) pada Sel 2.
3. Upload 11 PDF paper lewat Sel 3 (atau mount Google Drive).
4. Jalankan sel berikutnya berurutan → hasil akhir: `literature_gap_matrix.csv` dan `literature_gap_matrix.md`.

**Revisi 2026-10-26:** SEL 3 skip upload saat P01–P10 sudah tersedia; SEL 7 ekstraksi per field dengan retry, log dan checkpoint; SEL 8 menunda sintesis jika ada error; SEL 9 ZIP hasil. Semua keluaran tetap membutuhkan audit manual terhadap PDF.


In [1]:
# ============================================================
# SEL 1: Instalasi dependensi (Google Colab)
# ============================================================
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters \
    sentence-transformers faiss-cpu pypdf pandas tabulate
print("Instalasi selesai.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Instalasi selesai.


In [2]:
# ============================================================
# SEL 2: Konfigurasi API Key & Model (Menggunakan Model Groq Flagship Aktif)
# ============================================================
import os
from getpass import getpass

# Masukkan Groq API key (tidak ditampilkan di layar)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Masukkan GROQ API Key: ")

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # 768 dimensi, lokal
LLM_MODEL       = "openai/gpt-oss-120b"       # Menggunakan model sesuai permintaan pengguna
TEMPERATURE     = 0.1                          # meminimalkan deviasi generasi
CHUNK_SIZE      = 1000
CHUNK_OVERLAP   = 150
TOP_K           = 5
print(f"Konfigurasi diperbarui dengan model aktif: {LLM_MODEL}")

Masukkan GROQ API Key: ··········
Konfigurasi diperbarui dengan model aktif: openai/gpt-oss-120b


In [3]:
# SEL 3: Gunakan 10 PDF yang sudah tersedia; upload hanya bila belum lengkap
import os, glob, zipfile, shutil, re
from google.colab import files
os.makedirs("papers", exist_ok=True)

def current_pdfs():
    return sorted(glob.glob("papers/P[0-9][0-9]_*.pdf"))

def valid_corpus(paths):
    ids=[os.path.basename(p).split("_")[0] for p in paths]
    return len(paths)==10 and sorted(ids)==[f"P{i:02d}" for i in range(1,11)] and all(os.path.getsize(p)>0 for p in paths)

pdf_files=current_pdfs()
if not valid_corpus(pdf_files):
    print(f"Korpus belum lengkap ({len(pdf_files)}/10). Unggah ZIP final atau PDF yang belum ada.")
    uploaded=files.upload()
    for name in uploaded:
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                for item in z.infolist():
                    if item.is_dir() or not re.fullmatch(r"P(?:0[1-9]|10)_[^/\\]+\.pdf",os.path.basename(item.filename),re.I): continue
                    dest=os.path.join("papers",os.path.basename(item.filename))
                    with z.open(item) as src,open(dest,"wb") as dst: shutil.copyfileobj(src,dst)
        elif re.fullmatch(r"P(?:0[1-9]|10)_.+\.pdf",os.path.basename(name),re.I):
            shutil.move(name,os.path.join("papers",os.path.basename(name)))
    pdf_files=current_pdfs()
assert valid_corpus(pdf_files), f"Diperlukan tepat P01–P10, ditemukan: {[os.path.basename(p) for p in pdf_files]}"
print("Korpus valid, upload dilewati pada Run all berikutnya:")
for p in pdf_files: print("-",os.path.basename(p))

Korpus belum lengkap (0/10). Unggah ZIP final atau PDF yang belum ada.


Saving ReguLaw_RAG_10_Paper_Final.zip to ReguLaw_RAG_10_Paper_Final.zip
Korpus valid, upload dilewati pada Run all berikutnya:
- P01_Legal-DC_Benchmarking_RAG_for_Legal_Documents.pdf
- P02_Rhetorical_Role_Aware_RAG_for_Legal_QA.pdf
- P03_Legal_Query_RAG.pdf
- P04_Auditable_LLM_RAG_for_Financial_Documents.pdf
- P05_MultiFinRAG_Financial_Question_Answering.pdf
- P06_ChatLaw_Multi_Agent_Legal_Assistant.pdf
- P07_Regulatory_Compliance_QA_Financial_Services.pdf
- P08_Two_Stage_LLM_Regulatory_Retrieval_and_QA.pdf
- P09_GraphRAG_Indonesian_Legal_Question_Answering.pdf
- P10_Rethinking_Retrieval_Agentic_NonVector_Financial_RAG.pdf


In [4]:
# SEL 4: PDF -> page documents -> chunks, dengan provenance
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

docs=[]
for f in pdf_files:
    paper_id = os.path.basename(f).split("_")[0]
    assert paper_id.startswith("P") and paper_id[1:].isdigit(), f"Nama PDF perlu diawali P01_, P02_, dst: {f}"
    for d in PyPDFLoader(f).load():
        d.metadata.update({"paper_id":paper_id,"source_paper":os.path.basename(f),"page_number":int(d.metadata.get("page",0))+1})
        docs.append(d)
splitter=RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE,chunk_overlap=CHUNK_OVERLAP)
chunked_docs=splitter.split_documents(docs)
for i,d in enumerate(chunked_docs): d.metadata["chunk_id"]=f'{d.metadata["paper_id"]}_pg{d.metadata["page_number"]}_chunk{i}'
print("Halaman:",len(docs),"Chunks:",len(chunked_docs),"Per paper:",Counter(d.metadata["paper_id"] for d in chunked_docs))


/tmp/ipykernel_820/2805203594.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Halaman: 133 Chunks: 623 Per paper: Counter({'P06': 89, 'P04': 87, 'P03': 84, 'P01': 77, 'P07': 73, 'P09': 57, 'P05': 54, 'P10': 42, 'P02': 36, 'P08': 24})


In [5]:
# SEL 5: FAISS indeks TERPISAH per paper untuk isolasi retrieval yang ketat
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from collections import defaultdict
from tqdm.auto import tqdm
embedding_model=HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
groups=defaultdict(list)
for d in chunked_docs: groups[d.metadata["paper_id"]].append(d)
assert len(groups)==10
paper_indexes={}
for paper_id, paper_docs in tqdm(sorted(groups.items()),desc="Indeks FAISS per paper"):
    db=None
    for start in range(0,len(paper_docs),50):
        batch=paper_docs[start:start+50]
        if db is None: db=FAISS.from_documents(batch,embedding_model)
        else: db.add_documents(batch)
    paper_indexes[paper_id]=db

def retrieve_for_paper(question,paper_id,k=TOP_K):
    if paper_id not in paper_indexes: raise ValueError(f"paper_id tidak dikenal: {paper_id}")
    hits=paper_indexes[paper_id].similarity_search(question,k=k)
    assert all(d.metadata["paper_id"]==paper_id for d in hits),"KEBOCORAN antar paper!"
    return hits
print("FAISS per-paper siap:",sorted(paper_indexes))


/tmp/ipykernel_820/4152504600.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model=HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indeks FAISS per paper:   0%|          | 0/10 [00:00<?, ?it/s]

FAISS per-paper siap: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10']


In [6]:
# SEL 6: Grounded QA, wajib paper_id dan bukti halaman
from langchain_groq import ChatGroq
llm=ChatGroq(model_name=LLM_MODEL,temperature=TEMPERATURE)
def evidence_context(hits):
    return "\n\n".join(f'[paper_id={d.metadata["paper_id"]}; page={d.metadata["page_number"]}; chunk_id={d.metadata["chunk_id"]}]\n{d.page_content}' for d in hits)
def ask(question,paper_id):
    hits=retrieve_for_paper(question,paper_id)
    prompt=("Jawab hanya dari konteks PDF. Jika tidak ditemukan, katakan Tidak disebutkan eksplisit. "
            "Sebutkan nomor halaman untuk setiap klaim faktual.\n\n"+evidence_context(hits)+"\n\nPertanyaan: "+question)
    return llm.invoke(prompt).content.strip()
print("QA siap. Contoh: ask('Apa dataset yang digunakan?', 'P09')")


QA siap. Contoh: ask('Apa dataset yang digunakan?', 'P09')


In [7]:
# SEL 6b: Uji isolasi retrieval (tidak memerlukan panggilan Groq)
for pid in sorted(paper_indexes):
    hits=retrieve_for_paper("methods dataset experimental results limitations",pid)
    assert hits and all(d.metadata["paper_id"]==pid for d in hits)
    print(pid,"OK",[(d.metadata["page_number"],d.metadata["chunk_id"]) for d in hits[:2]])


P01 OK [(8, 'P01_pg8_chunk34'), (18, 'P01_pg18_chunk67')]
P02 OK [(11, 'P02_pg11_chunk106'), (3, 'P02_pg3_chunk85')]
P03 OK [(12, 'P03_pg12_chunk166'), (8, 'P03_pg8_chunk154')]
P04 OK [(10, 'P04_pg10_chunk240'), (17, 'P04_pg17_chunk273')]
P05 OK [(6, 'P05_pg6_chunk314'), (6, 'P05_pg6_chunk313')]
P06 OK [(1, 'P06_pg1_chunk344'), (14, 'P06_pg14_chunk420')]
P07 OK [(2, 'P07_pg2_chunk432'), (9, 'P07_pg9_chunk470')]
P08 OK [(3, 'P08_pg3_chunk511'), (4, 'P08_pg4_chunk517')]
P09 OK [(7, 'P09_pg7_chunk551'), (7, 'P09_pg7_chunk552')]
P10 OK [(3, 'P10_pg3_chunk593'), (1, 'P10_pg1_chunk582')]


In [8]:
# SEL 7: Ekstraksi tahan-error; retry per FIELD, checkpoint, audit halaman
import pandas as pd, json, re, os, time
from pathlib import Path
from langchain_core.messages import HumanMessage, SystemMessage
os.makedirs("outputs",exist_ok=True)
FIELDS=["DOI","Problem","Metode","Dataset","Temuan","Keterbatasan","Future Work"]
QUERIES={
 "DOI":"DOI publication journal conference arxiv title author bibliographic information",
 "Problem":"research problem motivation challenge gap introduction abstract",
 "Metode":"proposed method architecture approach pipeline model retriever algorithm",
 "Dataset":"dataset benchmark corpus number questions documents data source",
 "Temuan":"experimental results evaluation metric table scores comparison baseline",
 "Keterbatasan":"limitations threats to validity weaknesses evaluation restrictions discussion",
 "Future Work":"future work future research further improvements conclusion",
}
# Ekstraksi satu field per panggilan untuk menghindari JSON panjang yang terpotong.
# Setiap field menggunakan bukti dari PDF paper itu saja.
MAX_RETRIES=3

def get_hits(pid,field,k=8):
    questions=[QUERIES[field]]
    if field=="Temuan": questions += ["results table quantitative Recall MRR accuracy faithfulness win rate"]
    if field=="Keterbatasan": questions += ["limitations conclusion future work"]
    seen={};
    for question in questions:
        for d in retrieve_for_paper(question,pid,k=k):
            seen[d.metadata["chunk_id"]]=d
    return list(seen.values())

def parse_json_object(raw):
    text=raw.strip()
    text=re.sub(r"^```(?:json)?\s*|\s*```$","",text,flags=re.I).strip()
    try: return json.loads(text)
    except json.JSONDecodeError:
        start=text.find("{");end=text.rfind("}")
        if start>=0 and end>start: return json.loads(text[start:end+1])
        raise

def extract_field(pid,field,hits):
    allowed_pages=sorted({int(d.metadata["page_number"]) for d in hits})
    context=evidence_context(hits)
    sys=("Anda auditor literatur akademik. Jawab HANYA JSON object tanpa markdown: "
         '{"value":"ringkasan singkat dalam bahasa Indonesia","evidence_pages":[1,2]}. '
         "Jangan menebak DOI, angka, atau hasil eksperimen. "
         "Halaman evidence_pages harus benar-benar memuat klaim dan ada di konteks. "
         'Jika bukti tidak tersedia: {"value":"Tidak disebutkan eksplisit","evidence_pages":[]}. '
         "Untuk keterbatasan, hanya tulis yang secara eksplisit dinyatakan penulis; jangan menyimpulkan sendiri. "
         "Ringkas maksimal 100 kata.")
    question=f"Paper {pid}; field {field}. Kutipan sumber:\n{context}\nEkstrak field {field}."
    last_error=""
    for attempt in range(MAX_RETRIES):
        try:
            raw=llm.invoke([SystemMessage(content=sys),HumanMessage(content=question)]).content
            if not isinstance(raw,str): raise ValueError("LLM mengembalikan konten non-teks")
            item=parse_json_object(raw)
            if not isinstance(item,dict): raise ValueError("Bukan JSON object")
            val=item.get("value","Tidak disebutkan eksplisit")
            if not isinstance(val,str): raise ValueError("value bukan string")
            pages=item.get("evidence_pages",[])
            if not isinstance(pages,list): raise ValueError("evidence_pages bukan list")
            pages=[int(p) for p in pages]
            if any(p not in allowed_pages for p in pages): raise ValueError(f"Halaman tidak ada dalam evidence: {pages}")
            if val.strip() not in ("Tidak disebutkan eksplisit","") and not pages:
                raise ValueError("Klaim tanpa evidence_pages")
            if not val.strip(): val="Tidak disebutkan eksplisit"
            if val=="Tidak disebutkan eksplisit": pages=[]
            return val,pages,"PERLU VALIDASI MANUAL",raw
        except Exception as e:
            last_error=f"{type(e).__name__}: {e}"
            if attempt<MAX_RETRIES-1: time.sleep(2*(attempt+1))
    return "GAGAL EKSTRAKSI",[],f"ERROR: {last_error}",raw if 'raw' in locals() else ""

rows=[]; audit=[]; failures=[]
for idx,pdf in enumerate(pdf_files,1):
    pid=os.path.basename(pdf).split("_")[0]
    row={"No":idx,"paper_id":pid,"Paper":os.path.basename(pdf)}
    statuses=[]
    for field in FIELDS:
        hits=get_hits(pid,field)
        for d in hits:
            audit.append({"paper_id":pid,"field":field,"page":d.metadata["page_number"],
                          "chunk_id":d.metadata["chunk_id"],"text":d.page_content})
        value,pages,status,raw=extract_field(pid,field,hits)
        row[field]=value
        row[field+"_pages"]=', '.join(map(str,pages))
        statuses.append(status)
        if status.startswith("ERROR"):
            failures.append({"paper_id":pid,"field":field,"error":status,"raw_response":raw})
        print(pid,field,status)
    row["status"]="PERLU VALIDASI MANUAL" if not failures or all(f["paper_id"]!=pid for f in failures) else "EKSTRAKSI PARSIAL — CEK LOG"
    rows.append(row)
    # checkpoint setelah setiap paper agar hasil tidak hilang jika Colab terputus
    pd.DataFrame(rows).to_csv("outputs/literature_gap_matrix_checkpoint.csv",index=False,encoding="utf-8-sig")
    pd.DataFrame(failures).to_csv("outputs/extraction_errors.csv",index=False,encoding="utf-8-sig")
df=pd.DataFrame(rows)
evidence_df=pd.DataFrame(audit).drop_duplicates(subset=["paper_id","field","chunk_id"])
assert len(df)==10 and df.paper_id.is_unique
print("Selesai. Gagal per field:",len(failures))
display(df[["paper_id","status"]])

P01 DOI PERLU VALIDASI MANUAL
P01 Problem PERLU VALIDASI MANUAL
P01 Metode PERLU VALIDASI MANUAL
P01 Dataset PERLU VALIDASI MANUAL
P01 Temuan PERLU VALIDASI MANUAL
P01 Keterbatasan PERLU VALIDASI MANUAL
P01 Future Work PERLU VALIDASI MANUAL
P02 DOI PERLU VALIDASI MANUAL
P02 Problem PERLU VALIDASI MANUAL
P02 Metode PERLU VALIDASI MANUAL
P02 Dataset PERLU VALIDASI MANUAL
P02 Temuan PERLU VALIDASI MANUAL
P02 Keterbatasan PERLU VALIDASI MANUAL
P02 Future Work PERLU VALIDASI MANUAL
P03 DOI PERLU VALIDASI MANUAL
P03 Problem PERLU VALIDASI MANUAL
P03 Metode PERLU VALIDASI MANUAL
P03 Dataset PERLU VALIDASI MANUAL
P03 Temuan PERLU VALIDASI MANUAL
P03 Keterbatasan PERLU VALIDASI MANUAL
P03 Future Work PERLU VALIDASI MANUAL
P04 DOI PERLU VALIDASI MANUAL
P04 Problem PERLU VALIDASI MANUAL
P04 Metode PERLU VALIDASI MANUAL
P04 Dataset PERLU VALIDASI MANUAL
P04 Temuan PERLU VALIDASI MANUAL
P04 Keterbatasan PERLU VALIDASI MANUAL
P04 Future Work PERLU VALIDASI MANUAL
P05 DOI PERLU VALIDASI MANUAL
P05 Pr

,paper_id,status
0,P01,PERLU VALIDASI MANUAL
1,P02,PERLU VALIDASI MANUAL
2,P03,PERLU VALIDASI MANUAL
3,P04,PERLU VALIDASI MANUAL
4,P05,PERLU VALIDASI MANUAL
5,P06,PERLU VALIDASI MANUAL
6,P07,PERLU VALIDASI MANUAL
7,P08,EKSTRAKSI PARSIAL — CEK LOG
8,P09,EKSTRAKSI PARSIAL — CEK LOG
9,P10,EKSTRAKSI PARSIAL — CEK LOG


In [9]:
# SEL 8: Sintesis konservatif, jangan menyimpulkan jika ekstraksi belum lengkap
bad=df[df.status.str.contains("EKSTRAKSI PARSIAL",na=False)]
if len(bad):
    gap_analysis=("SINTESIS DITUNDA: terdapat ekstraksi gagal pada "
                  +", ".join(bad.paper_id.tolist())
                  +". Perbaiki sel 7 dan jalankan ulang sebelum menarik research gap.")
    print(gap_analysis)
else:
    matrix_context=df[["paper_id","Problem","Metode","Dataset","Temuan","Keterbatasan"]].to_json(orient="records",force_ascii=False)
    prompt=("Buat tepat 3 HIPOTESIS research gap ReguLaw-RAG dari ringkasan paper di bawah. "
            "Setiap hipotesis sertakan paper_id pendukung, bedakan keterbatasan eksplisit vs inferensi, "
            "jangan mengklaim belum ada studi sejenis secara absolut, jangan menciptakan nama atau nomor regulasi, "
            "jangan mengusulkan target angka tanpa justifikasi. Beri metrik evaluasi yang operasional. "
            "Ini sintesis awal dan tetap memerlukan pemeriksaan PDF manual.\n"+matrix_context)
    gap_analysis=llm.invoke(prompt).content.strip()
    print(gap_analysis)

SINTESIS DITUNDA: terdapat ekstraksi gagal pada P08, P09, P10. Perbaiki sel 7 dan jalankan ulang sebelum menarik research gap.


In [10]:
# SEL 9: Ekspor artefak dan log; unduh ZIP sekali saja
import os,zipfile
from google.colab import files
os.makedirs("outputs",exist_ok=True)
df.to_csv("outputs/literature_gap_matrix.csv",index=False,encoding="utf-8-sig")
evidence_df.to_csv("outputs/retrieved_evidence.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(failures,columns=["paper_id","field","error","raw_response"]).to_csv("outputs/extraction_errors.csv",index=False,encoding="utf-8-sig")
with open("outputs/literature_gap_matrix.md","w",encoding="utf-8") as f:
    f.write("# Literature GAP Matrix — otomatis, BELUM diverifikasi manual\n\n")
    f.write(df.fillna("").to_markdown(index=False))
    f.write("\n\n# Sintesis research gap (sementara)\n\n"+gap_analysis)
with zipfile.ZipFile("ReguLaw_RAG_Hasil_Eksperimen.zip","w",zipfile.ZIP_DEFLATED) as z:
    for name in ["literature_gap_matrix.csv","literature_gap_matrix.md","retrieved_evidence.csv","extraction_errors.csv"]:
        z.write(os.path.join("outputs",name),arcname=name)
print("Ekspor selesai:",len(df),"paper,",len(failures),"field error")
files.download("ReguLaw_RAG_Hasil_Eksperimen.zip")

Ekspor selesai: 10 paper, 20 field error


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>